In [1]:
# Logistic Regression using Gradient Descent
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


# ============================================================
# 2. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k Dataset (1).csv")

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 3. TARGET VARIABLE
# ============================================================

# PlacementStatus:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 4. SELECT NUMERICAL PREDICTORS
# ============================================================

features = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

X = df[features].copy()


# ============================================================
# 5. HANDLE MISSING VALUES
# ============================================================

for col in features:
    X[col] = X[col].fillna(X[col].median())


# ============================================================
# 6. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 7. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


# ============================================================
# 8. CONVERT TARGET TO NUMPY ARRAY
# ============================================================

y_train = y_train.to_numpy()

y_test = y_test.to_numpy()


# ============================================================
# 9. ADD INTERCEPT / BIAS COLUMN
# ============================================================

X_train_bias = np.c_[
    np.ones(X_train_scaled.shape[0]),
    X_train_scaled
]

X_test_bias = np.c_[
    np.ones(X_test_scaled.shape[0]),
    X_test_scaled
]


# ============================================================
# 10. SIGMOID FUNCTION
# ============================================================

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


# ============================================================
# 11. GRADIENT DESCENT FUNCTION
# ============================================================

def gradient_descent(
    X,
    y,
    learning_rate=0.01,
    n_iters=1000
):

    m, n = X.shape

    # Initialize parameters
    theta = np.zeros(n)

    # Store loss values
    losses = []

    # --------------------------------------------------------
    # Gradient Descent Iterations
    # --------------------------------------------------------

    for i in range(n_iters):

        # Linear combination
        z = X @ theta

        # Apply sigmoid function
        y_prob = sigmoid(z)

        # Error
        error = y_prob - y

        # Binary Cross-Entropy Loss
        loss = -np.mean(
            y * np.log(y_prob + 1e-15) +
            (1 - y) * np.log(1 - y_prob + 1e-15)
        )

        # Store loss
        losses.append(loss)

        # Gradient
        gradient = (1 / m) * (X.T @ error)

        # Update parameters
        theta = theta - learning_rate * gradient

    return theta, losses


# ============================================================
# 12. TRAIN MODEL USING GRADIENT DESCENT
# ============================================================

theta, losses = gradient_descent(
    X_train_bias,
    y_train,
    learning_rate=0.01,
    n_iters=1000
)


# ============================================================
# 13. DISPLAY PARAMETERS
# ============================================================

print("\n============================================")
print("GRADIENT DESCENT PARAMETERS")
print("============================================")

print("\nIntercept:")
print(theta[0])

print("\nCoefficients:")
print(theta[1:])


# ============================================================
# 14. PREDICTION
# ============================================================

# Calculate probability
y_probability = sigmoid(X_test_bias @ theta)

# Convert probability into class
y_pred = (y_probability >= 0.5).astype(int)


# ============================================================
# 15. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

conf_matrix = confusion_matrix(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_probability)


# ============================================================
# 16. DISPLAY RESULTS
# ============================================================

print("\n============================================")
print("LOGISTIC REGRESSION USING GRADIENT DESCENT")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC Score:")
print(round(roc_auc, 4))

Dataset Shape: (50000, 31)

Columns:
['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly']

GRADIENT DESCENT PARAMETERS

Intercept:
0.9218340405296209

Coefficients:
[0.18692332 0.1943728  0.1972286  0.20705217 0.21675698 0.2212277
 0.22863596 0.23233103 0.08209883 0.17994371 0.23876704 0.22889737
 0.18315402 0.22123384 0.0637426  0.2011494  0.3681597  0.1729619
 0.28404215 0.06533557]

LOGISTIC REGRESSION USING GRADIENT DESCENT

Accuracy:
0.8905

Confusion Matrix:
[[3076  353]
 [ 742 5829]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       